# Light clips for the review pages

Makes small copies of the 20 segment clips (640 px wide, no sound, under 10 MB per part) so Claude can put them inside the
Counter Review page. The page then plays them by itself; you no longer pick videos on the tablet.

Output: `MyDrive/Playbook/review/match_video_2/proxies/<segment>_p0.mp4`, `_p1.mp4` (each about 2.5 minutes).

**Runtime: CPU is enough**, about 10-15 minutes for the whole match.
**Runtime -> Run all**; do not edit cells. Finished parts are skipped, so you can run it again if it stops.

In [ ]:
# Cell 1 - Drive and inputs
import os, glob, math, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
MATCH = 'match_video_2'
CLIPS = f'{PBD}/video_downloads/{MATCH}_clips'
OUT = f'{PBD}/review/{MATCH}/proxies'
os.makedirs(OUT, exist_ok=True)
clips = sorted(glob.glob(f'{CLIPS}/*.mp4'))
print(len(clips), 'clips in', CLIPS)
if not clips:
    raise SystemExit(f'No clips found in {CLIPS}')

In [ ]:
# Cell 2 - Light copies, in parts of at most 155 s (about 8.5 MB each)
MAX_PART = 155
def duration(path):
    out = subprocess.check_output(['ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'csv=p=0', path])
    return float(out)
made = 0
for clip in clips:
    name = os.path.basename(clip)[:-4]
    dur = duration(clip)
    n = max(1, math.ceil(dur / MAX_PART))
    part = dur / n
    for i in range(n):
        out = f'{OUT}/{name}_p{i}.mp4'
        if os.path.exists(out) and os.path.getsize(out) > 0:
            continue
        tmp = '/content/part.mp4'
        subprocess.run(['ffmpeg', '-y', '-v', 'error', '-ss', f'{i * part:.3f}', '-i', clip, '-t', f'{part:.3f}',
                        '-vf', 'scale=640:-2', '-r', '30', '-c:v', 'libx264', '-preset', 'veryfast', '-profile:v', 'main',
                        '-b:v', '420k', '-maxrate', '450k', '-bufsize', '900k', '-pix_fmt', 'yuv420p', '-an',
                        '-movflags', '+faststart', tmp], check=True)
        shutil.copy(tmp, out); made += 1
        print(f'{name}_p{i}.mp4  starts at {i * part:.1f} s  {os.path.getsize(out) / 1e6:.1f} MB')
print('made', made, 'parts')

In [ ]:
# Cell 3 - Check
parts = sorted(glob.glob(f'{OUT}/*.mp4'))
big = [p for p in parts if os.path.getsize(p) > 9.8e6]
print(len(parts), 'parts,', f'{sum(os.path.getsize(p) for p in parts) / 1e6:.0f} MB in total')
print('too big:', [os.path.basename(p) for p in big] or 'none')
print(f'\nDone. Tell Claude the light clips are in {OUT}.')